In [ ]:
# для обучения с SparseCategoricalCrossentropy на 60 эпох
import tensorflow as tf
import numpy as np
import os, sys
import time
import kagglehub
import zipfile
import io
import pandas as pd
import chardet
import csv
import subprocess

# Обновление kagglehub (необязательно)
try:
    subprocess.check_call(['pip', 'install', '--upgrade', 'kagglehub'])
    print("Kagglehub updated successfully.")
except subprocess.CalledProcessError as e:
    print(f"Failed to update kagglehub: {e}")

# ----------------------------------------------------------------------
# Загрузка данных
# ----------------------------------------------------------------------
# Загрузка датасета с помощью kagglehub
dataset_path = kagglehub.dataset_download("trainingdatapro/generated-e-mail-spam")
print("Путь к файлам датасета:", dataset_path)

# Функция для загрузки текста из zip-архива
def load_text_from_zip(zip_file_path):
    with zipfile.ZipFile(zip_file_path, 'r') as zip_file:
        # Получаем имя первого CSV-файла в zip-архиве
        text_file_name = None
        for name in zip_file.namelist():
            if name.endswith(".csv"):
                text_file_name = name
                break
        if not text_file_name:
            raise Exception("CSV-файл не найден в zip-архиве")

        with zip_file.open(text_file_name, 'r') as f:
            df = pd.read_csv(f, encoding = 'latin1')
            # Объединяем все текстовые столбцы в одну строку
            text_columns = df.select_dtypes(include='object').columns.tolist()
            text = " ".join(df[text_columns].astype(str).fillna('').values.flatten())
    return text

# Функция для загрузки текста из локального текстового файла
def load_text_from_file(file_path):
    with open(file_path, 'r', encoding='utf-8', errors='ignore') as f:
        text = f.read()
    return text

def load_text_from_csv(csv_file_path, delimiter=','):
        try:
            with open(csv_file_path, 'rb') as f:
                raw_data = f.read()
                result = chardet.detect(raw_data)
                encoding = result['encoding']
            with open(csv_file_path, 'r', encoding=encoding, errors='ignore') as f:
                lines = [line.strip() for line in f]

            temp_file = "temp.csv"
            with open(temp_file, 'w', encoding=encoding, errors='ignore') as f:
                f.write('\n'.join(lines))

            try:
                # Попробуем прочитать с предположением, что нет заголовков
                df = pd.read_csv(temp_file, encoding=encoding, on_bad_lines='skip',
                                 delimiter=delimiter, header=None,
                                 skipinitialspace=True)
            except pd.errors.ParserError as e:
                print(f"Ошибка парсинга CSV (header=None): {e}")
                # Попробуем прочитать с предположением, что есть заголовки
                df = pd.read_csv(temp_file, encoding=encoding, on_bad_lines='skip',
                                 delimiter=delimiter, skipinitialspace=True)
            os.remove(temp_file)
        except Exception as e:
            print(f"Ошибка загрузки CSV с кодировкой '{encoding}' или 'latin1': {e}")
            df = pd.read_csv(csv_file_path, encoding='latin1', on_bad_lines='skip', delimiter=delimiter)
        # Объединяем все текстовые столбцы в одну строку
        text_columns = df.select_dtypes(include='object').columns.tolist()
        text = " ".join(df[text_columns].astype(str).fillna('').values.flatten())
        return text

# Загрузка текста
DELIMITER = ','
if os.path.isdir(dataset_path) or dataset_path.endswith(".zip"):
    if os.path.isdir(dataset_path):
        for file in os.listdir(dataset_path):
            if file.endswith(".csv"):
                csv_file_path = os.path.join(dataset_path, file)
                text = load_text_from_csv(csv_file_path, DELIMITER)
                break
    elif dataset_path.endswith(".zip"):
        text = load_text_from_zip(dataset_path)


# ----------------------------------------------------------------------
# Предварительная обработка текста
# ----------------------------------------------------------------------

# Обрабатываем текст
print(f'Длина текста: {len(text)} символов')
print(f'Первые 1000 символов: {text[:1000]}')
vocab = sorted(set(text))
print(f'Количество уникальных символов: {len(vocab)}')

# example_texts = ['abcdefg', 'xyz']
example_texts = text[11:]
print(example_texts[:100])
print(example_texts[-1000:])
chars = tf.strings.unicode_split(example_texts, input_encoding='UTF-8')
print(chars)

ids_from_chars = tf.keras.layers.StringLookup(vocabulary=list(vocab), mask_token=None)
ids = ids_from_chars(chars)

chars_from_ids = tf.keras.layers.StringLookup(vocabulary=ids_from_chars.get_vocabulary(), invert=True, mask_token=None)
chars = chars_from_ids(ids)



all_ids = ids_from_chars(tf.strings.unicode_split(text, 'UTF-8'))
print(all_ids)

ids_dataset = tf.data.Dataset.from_tensor_slices(all_ids)
seq_length = int(100/2)
sequences = ids_dataset.batch(seq_length + 1, drop_remainder=True)

def split_input_target(sequence):
    input_text = sequence[:-1]
    target_text = sequence[1:]
    return input_text, target_text

def text_from_ids(ids):
  return tf.strings.reduce_join(chars_from_ids(ids), axis=-1)

dataset = sequences.map(split_input_target)


BATCH_SIZE = int(64/2)
BUFFER_SIZE = 10000
dataset = (dataset.shuffle(BUFFER_SIZE).batch(BATCH_SIZE, drop_remainder=True).prefetch(tf.data.experimental.AUTOTUNE))

vocab_size = len(ids_from_chars.get_vocabulary())
embedding_dim = int(256/2)
rnn_units = int(1024/2)

# ----------------------------------------------------------------------
# Модель
# ----------------------------------------------------------------------

class MyModel(tf.keras.Model):
  def __init__(self, vocab_size, embedding_dim, rnn_units):
    super().__init__()
    self.embedding = tf.keras.layers.Embedding(vocab_size, embedding_dim)
    self.gru = tf.keras.layers.GRU(rnn_units,
                                   return_sequences=True,
                                   return_state=True)
    self.dense = tf.keras.layers.Dense(vocab_size)

  def call(self, inputs, states=None, return_state=False, training=False):
    x = inputs
    x = self.embedding(x, training=training)
    if states is None:
      states = self.gru.get_initial_state(tf.shape(x)[0])
    x, states = self.gru(x, initial_state=states, training=training)
    x = self.dense(x, training=training)

    if return_state:
      return x, states
    else:
      return x

# ----------------------------------------------------------------------
# Обучение и генерация текста
# ----------------------------------------------------------------------

# Функция для обучения модели и генерации текста
def train_and_generate(model, chars_from_ids, ids_from_chars, epochs, loss_fn, optimizer_class, gen_length, temperature):
    # Создаем новый экземпляр оптимизатора для каждой итерации
    optimizer = optimizer_class()
    # optimizer = tf.keras.optimizers.Adam()

    try:
        model.compile(optimizer=optimizer, loss=loss_fn)
    except Exception as e:
        print(f"Ошибка компиляции модели с {loss_fn.__name__} функцией потерь и {optimizer.__class__.__name__} оптимизатором: {e}")
        return

    checkpoint_dir = './training_checkpoints'
    checkpoint_prefix = os.path.join(checkpoint_dir, "ckpt_{epoch}.weights.h5")
    checkpoint_callback = tf.keras.callbacks.ModelCheckpoint(
        filepath=checkpoint_prefix,
        save_weights_only=True)

    model.fit(dataset, epochs=epochs, callbacks=[checkpoint_callback])


    one_step_model = OneStep(model, chars_from_ids, ids_from_chars, temperature)

    start = time.time()
    states = None
    next_char = tf.constant(['How are you? ']) # Начальный текст
    result = [next_char]

    for n in range(gen_length):
        next_char, states = one_step_model.generate_one_step(next_char, states=states)
        result.append(next_char)

    result = tf.strings.join(result)
    end = time.time()

    print("="*80)
    print(f"Результаты обучения и генерации:")
    print(f"  Количество эпох: {epochs}")
    print(f"  Оптимизатор: {optimizer.__class__.__name__}")
    print(f"  Длина сгенерированного текста: {gen_length} символов")
    print(f"  Температура: {temperature}")
    print(f"  Сгенерированный текст:\n{result[0].numpy().decode('utf-8')}")
    print("="*80)
    print(f'Время выполнения: {end - start:.2f} секунд\n')

# ----------------------------------------------------------------------
# Генерация текста пошагово
# ----------------------------------------------------------------------

class OneStep(tf.keras.Model):
  def __init__(self, model, chars_from_ids, ids_from_chars, temperature=1.0):
    super().__init__()
    self.temperature = temperature
    self.model = model
    self.chars_from_ids = chars_from_ids
    self.ids_from_chars = ids_from_chars

    # Маска для предотвращения генерации "[UNK]"
    skip_ids = self.ids_from_chars(['[UNK]'])[:, None]
    sparse_mask = tf.SparseTensor(
        values=[-float('inf')]*len(skip_ids),
        indices=skip_ids,
        dense_shape=[len(ids_from_chars.get_vocabulary())])
    self.prediction_mask = tf.sparse.to_dense(sparse_mask)

  @tf.function
  def generate_one_step(self, inputs, states=None):
    # Преобразуем строки в ID токенов
    input_chars = tf.strings.unicode_split(inputs, 'UTF-8')
    input_ids = self.ids_from_chars(input_chars).to_tensor()

    # Запускаем модель
    predicted_logits, states = self.model(inputs=input_ids, states=states,
                                          return_state=True)
    # Используем только последнее предсказание
    predicted_logits = predicted_logits[:, -1, :]
    predicted_logits = predicted_logits/self.temperature
    # Применяем маску для предотвращения генерации "[UNK]"
    predicted_logits = predicted_logits + self.prediction_mask

    # Сэмплируем выходные логиты для генерации ID токенов
    predicted_ids = tf.random.categorical(predicted_logits, num_samples=1)
    predicted_ids = tf.squeeze(predicted_ids, axis=-1)

    # Преобразуем ID токенов в символы
    predicted_chars = self.chars_from_ids(predicted_ids)

    # Возвращаем символы и состояние модели
    return predicted_chars, states

# ----------------------------------------------------------------------
# Параметры экспериментов
# ----------------------------------------------------------------------

# Список количества эпох
EPOCHS_LIST = [60]

# Список длин генерируемого текста
GEN_LENGTHS = [100, 1000]


# Список оптимизаторов
OPTIMIZERS = [
    tf.keras.optimizers.Adam,
    tf.keras.optimizers.SGD,
    tf.keras.optimizers.RMSprop
]

TEMPERATURES = [1.0]

LOSS_FUNCTION = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True) # Определение функции потерь


# Запускаем циклы обучения и генерации
for epochs in EPOCHS_LIST:
    print(f"Обучение с эпохами: {epochs}")
    # Создаем новую модель перед каждым набором эпох
    model = MyModel(
        vocab_size=vocab_size,
        embedding_dim=embedding_dim,
        rnn_units=rnn_units
    )
    # Создаем новые StringLookup слои
    ids_from_chars = tf.keras.layers.StringLookup(vocabulary=list(vocab), mask_token=None)
    chars_from_ids = tf.keras.layers.StringLookup(vocabulary=ids_from_chars.get_vocabulary(), invert=True, mask_token=None)


    for optimizer_class in OPTIMIZERS:
        for gen_length in GEN_LENGTHS:
            for temperature in TEMPERATURES:
                train_and_generate(model, chars_from_ids, ids_from_chars, epochs, LOSS_FUNCTION, optimizer_class, gen_length, temperature)


print(f"\n\nWarning: Please check the CSV file. pandas skipped some lines because the number of fields was not correct. These lines may contain data you want. It is advised that you fix the CSV data to not skip any lines for best results.\n")


Kagglehub updated successfully.


100%|██████████| 76.3k/76.3k [00:00<00:00, 41.6MB/s]

Extracting files...
Путь к файлам датасета: /root/.cache/kagglehub/datasets/trainingdatapro/generated-e-mail-spam/versions/1


Длина текста: 24053 символов
Первые 1000 символов: Title	Text The Love-Booster Crew" Unbelievable Simple Trick to Help Make Your Life Easier	"Yo! What's up! Do you ever feel like you just can't seem to catch a break? Are you always drowning in work and responsibilities? BOB'S MAGIC is ridiculously simple to use and helps you smash through your to-do list in no time. Say goodbye to stress and hello to productivity with this game-changing trick! The BOB'S MAGIC Crew" Find your ride or die today and have an epic summer full of unforgettable memories! The YourLuv4Ever.com Crew" Visit <LINK> today to learn more about our program and how it can work for you. Take the first step towards a brighter financial future. Don't miss out on this incredible opportunity! Sign up now for a free consultation and start your journey towards financial freedom. The EraseDebtToday Team" Last Call - Hurry Up!	"Yo dude! The Crew" Tip 1: Discover exciting new experiences - Trying out new activities together is a

В ходе обучения:
- Модель показала снижение функции потерь на протяжении 60 эпох, что свидетельствует об улучшении способности предсказывать последовательности символов.
- Adam, как правило, сходился быстрее и давал более связный текст на 1000 символов, в то время как SGD иногда генерировал более непредсказуемый текст.
- Длина текста: Модель способна генерировать как короткие, так и длинные тексты, что свидетельствует о её способности захватывать как локальные, так и более длинные зависимости в тексте.


In [ ]:
# для обучения с Sequence-to-Sequence Loss (with Masking) 60 эпох
import tensorflow as tf
import numpy as np
import os, sys
import time
import kagglehub
import zipfile
import io
import pandas as pd
import chardet
import csv
import subprocess

# Обновление kagglehub (необязательно)
try:
    subprocess.check_call(['pip', 'install', '--upgrade', 'kagglehub'])
    print("Kagglehub updated successfully.")
except subprocess.CalledProcessError as e:
    print(f"Failed to update kagglehub: {e}")

# ----------------------------------------------------------------------
# Загрузка данных
# ----------------------------------------------------------------------
# Загрузка датасета с помощью kagglehub
dataset_path = kagglehub.dataset_download("trainingdatapro/generated-e-mail-spam")
print("Путь к файлам датасета:", dataset_path)

# Функция для загрузки текста из zip-архива
def load_text_from_zip(zip_file_path):
    with zipfile.ZipFile(zip_file_path, 'r') as zip_file:
        # Получаем имя первого CSV-файла в zip-архиве
        text_file_name = None
        for name in zip_file.namelist():
            if name.endswith(".csv"):
                text_file_name = name
                break
        if not text_file_name:
            raise Exception("CSV-файл не найден в zip-архиве")

        with zip_file.open(text_file_name, 'r') as f:
            df = pd.read_csv(f, encoding = 'latin1')
            # Объединяем все текстовые столбцы в одну строку
            text_columns = df.select_dtypes(include='object').columns.tolist()
            text = " ".join(df[text_columns].astype(str).fillna('').values.flatten())
    return text

# Функция для загрузки текста из локального текстового файла
def load_text_from_file(file_path):
    with open(file_path, 'r', encoding='utf-8', errors='ignore') as f:
        text = f.read()
    return text

def load_text_from_csv(csv_file_path, delimiter=','):
        try:
            with open(csv_file_path, 'rb') as f:
                raw_data = f.read()
                result = chardet.detect(raw_data)
                encoding = result['encoding']
            with open(csv_file_path, 'r', encoding=encoding, errors='ignore') as f:
                lines = [line.strip() for line in f]

            temp_file = "temp.csv"
            with open(temp_file, 'w', encoding=encoding, errors='ignore') as f:
                f.write('\n'.join(lines))

            try:
                # Попробуем прочитать с предположением, что нет заголовков
                df = pd.read_csv(temp_file, encoding=encoding, on_bad_lines='skip',
                                 delimiter=delimiter, header=None,
                                 skipinitialspace=True)
            except pd.errors.ParserError as e:
                print(f"Ошибка парсинга CSV (header=None): {e}")
                # Попробуем прочитать с предположением, что есть заголовки
                df = pd.read_csv(temp_file, encoding=encoding, on_bad_lines='skip',
                                 delimiter=delimiter, skipinitialspace=True)
            os.remove(temp_file)
        except Exception as e:
            print(f"Ошибка загрузки CSV с кодировкой '{encoding}' или 'latin1': {e}")
            df = pd.read_csv(csv_file_path, encoding='latin1', on_bad_lines='skip', delimiter=delimiter)
        # Объединяем все текстовые столбцы в одну строку
        text_columns = df.select_dtypes(include='object').columns.tolist()
        text = " ".join(df[text_columns].astype(str).fillna('').values.flatten())
        return text

# Загрузка текста
DELIMITER = ','
if os.path.isdir(dataset_path) or dataset_path.endswith(".zip"):
    if os.path.isdir(dataset_path):
        for file in os.listdir(dataset_path):
            if file.endswith(".csv"):
                csv_file_path = os.path.join(dataset_path, file)
                text = load_text_from_csv(csv_file_path, DELIMITER)
                break
    elif dataset_path.endswith(".zip"):
        text = load_text_from_zip(dataset_path)


# ----------------------------------------------------------------------
# Предварительная обработка текста
# ----------------------------------------------------------------------

# Обрабатываем текст
print(f'Длина текста: {len(text)} символов')
print(f'Первые 1000 символов: {text[:1000]}')
vocab = sorted(set(text))
print(f'Количество уникальных символов: {len(vocab)}')

# example_texts = ['abcdefg', 'xyz']
example_texts = text[11:]
print(example_texts[:100])
print(example_texts[-1000:])
chars = tf.strings.unicode_split(example_texts, input_encoding='UTF-8')
print(chars)

ids_from_chars = tf.keras.layers.StringLookup(vocabulary=list(vocab), mask_token=None)
ids = ids_from_chars(chars)

chars_from_ids = tf.keras.layers.StringLookup(vocabulary=ids_from_chars.get_vocabulary(), invert=True, mask_token=None)
chars = chars_from_ids(ids)



all_ids = ids_from_chars(tf.strings.unicode_split(text, 'UTF-8'))
print(all_ids)

ids_dataset = tf.data.Dataset.from_tensor_slices(all_ids)
seq_length = int(100/2)
sequences = ids_dataset.batch(seq_length + 1, drop_remainder=True)

def split_input_target(sequence):
    input_text = sequence[:-1]
    target_text = sequence[1:]
    return input_text, target_text

def text_from_ids(ids):
  return tf.strings.reduce_join(chars_from_ids(ids), axis=-1)

dataset = sequences.map(split_input_target)


BATCH_SIZE = int(64/2)
BUFFER_SIZE = 10000
dataset = (dataset.shuffle(BUFFER_SIZE).batch(BATCH_SIZE, drop_remainder=True).prefetch(tf.data.experimental.AUTOTUNE))

vocab_size = len(ids_from_chars.get_vocabulary())
embedding_dim = int(256/2)
rnn_units = int(1024/2)

# ----------------------------------------------------------------------
# Модель
# ----------------------------------------------------------------------

class MyModel(tf.keras.Model):
  def __init__(self, vocab_size, embedding_dim, rnn_units):
    super().__init__()
    self.embedding = tf.keras.layers.Embedding(vocab_size, embedding_dim)
    self.gru = tf.keras.layers.GRU(rnn_units,
                                   return_sequences=True,
                                   return_state=True)
    self.dense = tf.keras.layers.Dense(vocab_size)

  def call(self, inputs, states=None, return_state=False, training=False):
    x = inputs
    x = self.embedding(x, training=training)
    if states is None:
      states = self.gru.get_initial_state(tf.shape(x)[0])
    x, states = self.gru(x, initial_state=states, training=training)
    x = self.dense(x, training=training)

    if return_state:
      return x, states
    else:
      return x

# ----------------------------------------------------------------------
# Обучение и генерация текста
# ----------------------------------------------------------------------

# Функция для обучения модели и генерации текста
def train_and_generate(model, chars_from_ids, ids_from_chars, epochs, loss_fn, optimizer_class, gen_length, temperature):
    # Создаем новый экземпляр оптимизатора для каждой итерации
    optimizer = optimizer_class()

    try:
        model.compile(optimizer=optimizer, loss=loss_fn)
    except Exception as e:
        print(f"Ошибка компиляции модели с {loss_fn.__name__} функцией потерь и {optimizer.__class__.__name__} оптимизатором: {e}")
        return

    checkpoint_dir = './training_checkpoints'
    checkpoint_prefix = os.path.join(checkpoint_dir, "ckpt_{epoch}.weights.h5")
    checkpoint_callback = tf.keras.callbacks.ModelCheckpoint(
        filepath=checkpoint_prefix,
        save_weights_only=True)

    model.fit(dataset, epochs=epochs, callbacks=[checkpoint_callback])


    one_step_model = OneStep(model, chars_from_ids, ids_from_chars, temperature)

    start = time.time()
    states = None
    next_char = tf.constant(['How are you? ']) # Начальный текст
    result = [next_char]

    for n in range(gen_length):
        next_char, states = one_step_model.generate_one_step(next_char, states=states)
        result.append(next_char)

    result = tf.strings.join(result)
    end = time.time()

    print("="*80)
    print(f"Результаты обучения и генерации:")
    print(f"  Количество эпох: {epochs}")
    print(f"  Оптимизатор: {optimizer.__class__.__name__}")
    print(f"  Длина сгенерированного текста: {gen_length} символов")
    print(f"  Температура: {temperature}")
    print(f"  Сгенерированный текст:\n{result[0].numpy().decode('utf-8')}")
    print("="*80)
    print(f'Время выполнения: {end - start:.2f} секунд\n')

# ----------------------------------------------------------------------
# Генерация текста пошагово
# ----------------------------------------------------------------------

class OneStep(tf.keras.Model):
  def __init__(self, model, chars_from_ids, ids_from_chars, temperature=1.0):
    super().__init__()
    self.temperature = temperature
    self.model = model
    self.chars_from_ids = chars_from_ids
    self.ids_from_chars = ids_from_chars

    # Маска для предотвращения генерации "[UNK]"
    skip_ids = self.ids_from_chars(['[UNK]'])[:, None]
    sparse_mask = tf.SparseTensor(
        values=[-float('inf')]*len(skip_ids),
        indices=skip_ids,
        dense_shape=[len(ids_from_chars.get_vocabulary())])
    self.prediction_mask = tf.sparse.to_dense(sparse_mask)

  @tf.function
  def generate_one_step(self, inputs, states=None):
    # Преобразуем строки в ID токенов
    input_chars = tf.strings.unicode_split(inputs, 'UTF-8')
    input_ids = self.ids_from_chars(input_chars).to_tensor()

    # Запускаем модель
    predicted_logits, states = self.model(inputs=input_ids, states=states,
                                          return_state=True)
    # Используем только последнее предсказание
    predicted_logits = predicted_logits[:, -1, :]
    predicted_logits = predicted_logits/self.temperature
    # Применяем маску для предотвращения генерации "[UNK]"
    predicted_logits = predicted_logits + self.prediction_mask

    # Сэмплируем выходные логиты для генерации ID токенов
    predicted_ids = tf.random.categorical(predicted_logits, num_samples=1)
    predicted_ids = tf.squeeze(predicted_ids, axis=-1)

    # Преобразуем ID токенов в символы
    predicted_chars = self.chars_from_ids(predicted_ids)

    # Возвращаем символы и состояние модели
    return predicted_chars, states


# ----------------------------------------------------------------------
# Кастомная функция потерь для Sequence-to-Sequence с маскированием
# ----------------------------------------------------------------------

def masked_loss(y_true, y_pred):
    mask = tf.math.logical_not(tf.math.equal(y_true, 0))
    loss_object = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True, reduction='none')
    loss = loss_object(y_true, y_pred)

    mask = tf.cast(mask, dtype=loss.dtype)
    loss *= mask
    return tf.reduce_mean(loss)

# ----------------------------------------------------------------------
# Параметры экспериментов
# ----------------------------------------------------------------------

# Список количества эпох
EPOCHS_LIST = [60]

# Список длин генерируемого текста
GEN_LENGTHS = [100, 1000]


# Список оптимизаторов
OPTIMIZERS = [
    tf.keras.optimizers.Adam,
    tf.keras.optimizers.SGD,
    tf.keras.optimizers.RMSprop
]

TEMPERATURES = [1.0]

# Sequence-to-Sequence Loss with Masking
LOSS_FUNCTION = masked_loss

# Запускаем циклы обучения и генерации
for epochs in EPOCHS_LIST:
    print(f"Обучение с эпохами: {epochs}")
    # Создаем новую модель перед каждым набором эпох
    model = MyModel(
        vocab_size=vocab_size,
        embedding_dim=embedding_dim,
        rnn_units=rnn_units
    )
     # Создаем новые StringLookup слои
    ids_from_chars = tf.keras.layers.StringLookup(vocabulary=list(vocab), mask_token=None)
    chars_from_ids = tf.keras.layers.StringLookup(vocabulary=ids_from_chars.get_vocabulary(), invert=True, mask_token=None)
    for optimizer_class in OPTIMIZERS:
        for gen_length in GEN_LENGTHS:
            for temperature in TEMPERATURES:
                train_and_generate(model, chars_from_ids, ids_from_chars, epochs, LOSS_FUNCTION, optimizer_class, gen_length, temperature)


print(f"\n\nWarning: Please check the CSV file. pandas skipped some lines because the number of fields was not correct. These lines may contain data you want. It is advised that you fix the CSV data to not skip any lines for best results.\n")


Kagglehub updated successfully.
Путь к файлам датасета: /root/.cache/kagglehub/datasets/trainingdatapro/generated-e-mail-spam/versions/1
Длина текста: 24053 символов
Первые 1000 символов: Title	Text The Love-Booster Crew" Unbelievable Simple Trick to Help Make Your Life Easier	"Yo! What's up! Do you ever feel like you just can't seem to catch a break? Are you always drowning in work and responsibilities? BOB'S MAGIC is ridiculously simple to use and helps you smash through your to-do list in no time. Say goodbye to stress and hello to productivity with this game-changing trick! The BOB'S MAGIC Crew" Find your ride or die today and have an epic summer full of unforgettable memories! The YourLuv4Ever.com Crew" Visit <LINK> today to learn more about our program and how it can work for you. Take the first step towards a brighter financial future. Don't miss out on this incredible opportunity! Sign up now for a free consultation and start your journey towards financial freedom. The EraseDeb

Модель обучена с использованием кастомной функции потерь masked_loss для sequence-to-sequence обучения с маскированием, что позволяет эффективно обрабатывать последовательности разной длины и учитывать нулевые маскирующие токены.

- Обученная модель демонстрирует способность генерировать текст, хотя и с ограниченной осмысленностью и связностью.
- Значения loss уменьшаются в процессе обучения, что указывает на то, что модель обучается минимизировать ошибку предсказания.
- Оптимизатор Adam, в основном, показывает наименьшие значения loss по итогам обучения.
- Сгенерированный текст может включать бессмысленные символьные последовательности, что, вероятно, связано с недостаточным размером обучающих данных, "неглубокой" моделью, отсутствием токенизации на уровне слов и прочими параметрами обучения.

In [ ]:
# для обучения с Negative Log Likelihood (NLL) 60 эпох
import tensorflow as tf
import numpy as np
import os, sys
import time
import kagglehub
import zipfile
import io
import pandas as pd
import chardet
import csv
import subprocess

# Обновление kagglehub (необязательно)
try:
    subprocess.check_call(['pip', 'install', '--upgrade', 'kagglehub'])
    print("Kagglehub updated successfully.")
except subprocess.CalledProcessError as e:
    print(f"Failed to update kagglehub: {e}")

# ----------------------------------------------------------------------
# Загрузка данных
# ----------------------------------------------------------------------
# Загрузка датасета с помощью kagglehub
dataset_path = kagglehub.dataset_download("trainingdatapro/generated-e-mail-spam")
print("Путь к файлам датасета:", dataset_path)

# Функция для загрузки текста из zip-архива
def load_text_from_zip(zip_file_path):
    with zipfile.ZipFile(zip_file_path, 'r') as zip_file:
        # Получаем имя первого CSV-файла в zip-архиве
        text_file_name = None
        for name in zip_file.namelist():
            if name.endswith(".csv"):
                text_file_name = name
                break
        if not text_file_name:
            raise Exception("CSV-файл не найден в zip-архиве")

        with zip_file.open(text_file_name, 'r') as f:
            df = pd.read_csv(f, encoding = 'latin1')
            # Объединяем все текстовые столбцы в одну строку
            text_columns = df.select_dtypes(include='object').columns.tolist()
            text = " ".join(df[text_columns].astype(str).fillna('').values.flatten())
    return text

# Функция для загрузки текста из локального текстового файла
def load_text_from_file(file_path):
    with open(file_path, 'r', encoding='utf-8', errors='ignore') as f:
        text = f.read()
    return text

def load_text_from_csv(csv_file_path, delimiter=','):
        try:
            with open(csv_file_path, 'rb') as f:
                raw_data = f.read()
                result = chardet.detect(raw_data)
                encoding = result['encoding']
            with open(csv_file_path, 'r', encoding=encoding, errors='ignore') as f:
                lines = [line.strip() for line in f]

            temp_file = "temp.csv"
            with open(temp_file, 'w', encoding=encoding, errors='ignore') as f:
                f.write('\n'.join(lines))

            try:
                # Попробуем прочитать с предположением, что нет заголовков
                df = pd.read_csv(temp_file, encoding=encoding, on_bad_lines='skip',
                                 delimiter=delimiter, header=None,
                                 skipinitialspace=True)
            except pd.errors.ParserError as e:
                print(f"Ошибка парсинга CSV (header=None): {e}")
                # Попробуем прочитать с предположением, что есть заголовки
                df = pd.read_csv(temp_file, encoding=encoding, on_bad_lines='skip',
                                 delimiter=delimiter, skipinitialspace=True)
            os.remove(temp_file)
        except Exception as e:
            print(f"Ошибка загрузки CSV с кодировкой '{encoding}' или 'latin1': {e}")
            df = pd.read_csv(csv_file_path, encoding='latin1', on_bad_lines='skip', delimiter=delimiter)
        # Объединяем все текстовые столбцы в одну строку
        text_columns = df.select_dtypes(include='object').columns.tolist()
        text = " ".join(df[text_columns].astype(str).fillna('').values.flatten())
        return text

# Загрузка текста
DELIMITER = ','
if os.path.isdir(dataset_path) or dataset_path.endswith(".zip"):
    if os.path.isdir(dataset_path):
        for file in os.listdir(dataset_path):
            if file.endswith(".csv"):
                csv_file_path = os.path.join(dataset_path, file)
                text = load_text_from_csv(csv_file_path, DELIMITER)
                break
    elif dataset_path.endswith(".zip"):
        text = load_text_from_zip(dataset_path)


# ----------------------------------------------------------------------
# Предварительная обработка текста
# ----------------------------------------------------------------------

# Обрабатываем текст
print(f'Длина текста: {len(text)} символов')
print(f'Первые 1000 символов: {text[:1000]}')
vocab = sorted(set(text))
print(f'Количество уникальных символов: {len(vocab)}')

# example_texts = ['abcdefg', 'xyz']
example_texts = text[11:]
print(example_texts[:100])
print(example_texts[-1000:])
chars = tf.strings.unicode_split(example_texts, input_encoding='UTF-8')
print(chars)

ids_from_chars = tf.keras.layers.StringLookup(vocabulary=list(vocab), mask_token=None)
ids = ids_from_chars(chars)

chars_from_ids = tf.keras.layers.StringLookup(vocabulary=ids_from_chars.get_vocabulary(), invert=True, mask_token=None)
chars = chars_from_ids(ids)



all_ids = ids_from_chars(tf.strings.unicode_split(text, 'UTF-8'))
print(all_ids)

ids_dataset = tf.data.Dataset.from_tensor_slices(all_ids)
seq_length = int(100/2)
sequences = ids_dataset.batch(seq_length + 1, drop_remainder=True)

def split_input_target(sequence):
    input_text = sequence[:-1]
    target_text = sequence[1:]
    return input_text, target_text

def text_from_ids(ids):
  return tf.strings.reduce_join(chars_from_ids(ids), axis=-1)

dataset = sequences.map(split_input_target)


BATCH_SIZE = int(64/2)
BUFFER_SIZE = 10000
dataset = (dataset.shuffle(BUFFER_SIZE).batch(BATCH_SIZE, drop_remainder=True).prefetch(tf.data.experimental.AUTOTUNE))

vocab_size = len(ids_from_chars.get_vocabulary())
embedding_dim = int(256/2)
rnn_units = int(1024/2)

# ----------------------------------------------------------------------
# Модель
# ----------------------------------------------------------------------

class MyModel(tf.keras.Model):
  def __init__(self, vocab_size, embedding_dim, rnn_units):
    super().__init__()
    self.embedding = tf.keras.layers.Embedding(vocab_size, embedding_dim)
    self.gru = tf.keras.layers.GRU(rnn_units,
                                   return_sequences=True,
                                   return_state=True)
    self.dense = tf.keras.layers.Dense(vocab_size)

  def call(self, inputs, states=None, return_state=False, training=False):
    x = inputs
    x = self.embedding(x, training=training)
    if states is None:
      states = self.gru.get_initial_state(tf.shape(x)[0])
    x, states = self.gru(x, initial_state=states, training=training)
    x = self.dense(x, training=training)

    if return_state:
      return x, states
    else:
      return x

# ----------------------------------------------------------------------
# Обучение и генерация текста
# ----------------------------------------------------------------------

# Функция для обучения модели и генерации текста
def train_and_generate(model, chars_from_ids, ids_from_chars, epochs, loss_fn, optimizer_class, gen_length, temperature):
    # Создаем новый экземпляр оптимизатора для каждой итерации
    optimizer = optimizer_class()

    try:
        model.compile(optimizer=optimizer, loss=loss_fn)
    except Exception as e:
        print(f"Ошибка компиляции модели с {loss_fn.__name__} функцией потерь и {optimizer.__class__.__name__} оптимизатором: {e}")
        return

    checkpoint_dir = './training_checkpoints'
    checkpoint_prefix = os.path.join(checkpoint_dir, "ckpt_{epoch}.weights.h5")
    checkpoint_callback = tf.keras.callbacks.ModelCheckpoint(
        filepath=checkpoint_prefix,
        save_weights_only=True)

    model.fit(dataset, epochs=epochs, callbacks=[checkpoint_callback])


    one_step_model = OneStep(model, chars_from_ids, ids_from_chars, temperature)

    start = time.time()
    states = None
    next_char = tf.constant(['How are you? ']) # Начальный текст
    result = [next_char]

    for n in range(gen_length):
        next_char, states = one_step_model.generate_one_step(next_char, states=states)
        result.append(next_char)

    result = tf.strings.join(result)
    end = time.time()

    print("="*80)
    print(f"Результаты обучения и генерации:")
    print(f"  Количество эпох: {epochs}")
    print(f"  Длина сгенерированного текста: {gen_length} символов")
    print(f"  Температура: {temperature}")
    print(f"  Сгенерированный текст:\n{result[0].numpy().decode('utf-8')}")
    print("="*80)
    print(f'Время выполнения: {end - start:.2f} секунд\n')

# ----------------------------------------------------------------------
# Генерация текста пошагово
# ----------------------------------------------------------------------

class OneStep(tf.keras.Model):
  def __init__(self, model, chars_from_ids, ids_from_chars, temperature=1.0):
    super().__init__()
    self.temperature = temperature
    self.model = model
    self.chars_from_ids = chars_from_ids
    self.ids_from_chars = ids_from_chars

    # Маска для предотвращения генерации "[UNK]"
    skip_ids = self.ids_from_chars(['[UNK]'])[:, None]
    sparse_mask = tf.SparseTensor(
        values=[-float('inf')]*len(skip_ids),
        indices=skip_ids,
        dense_shape=[len(ids_from_chars.get_vocabulary())])
    self.prediction_mask = tf.sparse.to_dense(sparse_mask)

  @tf.function
  def generate_one_step(self, inputs, states=None):
    # Преобразуем строки в ID токенов
    input_chars = tf.strings.unicode_split(inputs, 'UTF-8')
    input_ids = self.ids_from_chars(input_chars).to_tensor()

    # Запускаем модель
    predicted_logits, states = self.model(inputs=input_ids, states=states,
                                          return_state=True)
    # Используем только последнее предсказание
    predicted_logits = predicted_logits[:, -1, :]
    predicted_logits = predicted_logits/self.temperature
    # Применяем маску для предотвращения генерации "[UNK]"
    predicted_logits = predicted_logits + self.prediction_mask

    # Сэмплируем выходные логиты для генерации ID токенов
    predicted_ids = tf.random.categorical(predicted_logits, num_samples=1)
    predicted_ids = tf.squeeze(predicted_ids, axis=-1)

    # Преобразуем ID токенов в символы
    predicted_chars = self.chars_from_ids(predicted_ids)

    # Возвращаем символы и состояние модели
    return predicted_chars, states


# ----------------------------------------------------------------------
# Параметры экспериментов
# ----------------------------------------------------------------------

# Список количества эпох
EPOCHS_LIST = [60]

# Список длин генерируемого текста
GEN_LENGTHS = [100, 1000]


# Список оптимизаторов
OPTIMIZERS = [
    tf.keras.optimizers.Adam,
    tf.keras.optimizers.SGD,
    tf.keras.optimizers.RMSprop
]

TEMPERATURES = [1.0]

# NLL (SparseCategoricalCrossentropy)
LOSS_FUNCTION = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True)

# Запускаем циклы обучения и генерации
for epochs in EPOCHS_LIST:
    print(f"Обучение с эпохами: {epochs}")
    # Создаем новую модель перед каждым набором эпох
    model = MyModel(
        vocab_size=vocab_size,
        embedding_dim=embedding_dim,
        rnn_units=rnn_units
    )
    # Создаем новые StringLookup слои
    ids_from_chars = tf.keras.layers.StringLookup(vocabulary=list(vocab), mask_token=None)
    chars_from_ids = tf.keras.layers.StringLookup(vocabulary=ids_from_chars.get_vocabulary(), invert=True, mask_token=None)

    for optimizer_class in OPTIMIZERS:
        for gen_length in GEN_LENGTHS:
            for temperature in TEMPERATURES:
                train_and_generate(model, chars_from_ids, ids_from_chars, epochs, LOSS_FUNCTION, optimizer_class, gen_length, temperature)


print(f"\n\nWarning: Please check the CSV file. pandas skipped some lines because the number of fields was not correct. These lines may contain data you want. It is advised that you fix the CSV data to not skip any lines for best results.\n")


Kagglehub updated successfully.
Путь к файлам датасета: /root/.cache/kagglehub/datasets/trainingdatapro/generated-e-mail-spam/versions/1
Длина текста: 24053 символов
Первые 1000 символов: Title	Text The Love-Booster Crew" Unbelievable Simple Trick to Help Make Your Life Easier	"Yo! What's up! Do you ever feel like you just can't seem to catch a break? Are you always drowning in work and responsibilities? BOB'S MAGIC is ridiculously simple to use and helps you smash through your to-do list in no time. Say goodbye to stress and hello to productivity with this game-changing trick! The BOB'S MAGIC Crew" Find your ride or die today and have an epic summer full of unforgettable memories! The YourLuv4Ever.com Crew" Visit <LINK> today to learn more about our program and how it can work for you. Take the first step towards a brighter financial future. Don't miss out on this incredible opportunity! Sign up now for a free consultation and start your journey towards financial freedom. The EraseDeb

- Наблюдается устойчивое снижение функции потерь (loss) во всех случаях, что свидетельствует об обучении модели.
- Сгенерированный текст показывает некоторую связность, особенно в коротких предложениях, но в целом заметны повторения и грамматические ошибки.
- Наблюдаются различия в сгенерированном тексте в зависимости от использованного оптимизатора, но в целом качество текста остается на схожем уровне. Однако, длинные сгенерированные тексты (1000 символов) наиболее подвержены повторениям и несут меньше смысла, чем короткие (100 символов).


Общий вывод по 3 эк-м:
Генерация осмысленного текста — достаточно сложная задача, и для ее решения требуется более сложная архитектура и больший объем данных.
Однако, на данном этапе, в работах обучение проходит хорошо, функции потерь находятся на уровне ~ 0.06-0.07, поэтому генерируется относительно связный понятный текст.

In [ ]:
# для обучения с SparseCategoricalCrossentropy
import tensorflow as tf
import numpy as np
import os, sys
import time
import kagglehub
import zipfile
import io
import pandas as pd
import chardet
import csv
import subprocess

# Обновление kagglehub (необязательно)
try:
    subprocess.check_call(['pip', 'install', '--upgrade', 'kagglehub'])
    print("Kagglehub updated successfully.")
except subprocess.CalledProcessError as e:
    print(f"Failed to update kagglehub: {e}")

# ----------------------------------------------------------------------
# Загрузка данных
# ----------------------------------------------------------------------
# Загрузка датасета с помощью kagglehub
dataset_path = kagglehub.dataset_download("trainingdatapro/generated-e-mail-spam")
print("Путь к файлам датасета:", dataset_path)

# Функция для загрузки текста из zip-архива
def load_text_from_zip(zip_file_path):
    with zipfile.ZipFile(zip_file_path, 'r') as zip_file:
        # Получаем имя первого CSV-файла в zip-архиве
        text_file_name = None
        for name in zip_file.namelist():
            if name.endswith(".csv"):
                text_file_name = name
                break
        if not text_file_name:
            raise Exception("CSV-файл не найден в zip-архиве")

        with zip_file.open(text_file_name, 'r') as f:
            df = pd.read_csv(f, encoding = 'latin1')
            # Объединяем все текстовые столбцы в одну строку
            text_columns = df.select_dtypes(include='object').columns.tolist()
            text = " ".join(df[text_columns].astype(str).fillna('').values.flatten())
    return text

# Функция для загрузки текста из локального текстового файла
def load_text_from_file(file_path):
    with open(file_path, 'r', encoding='utf-8', errors='ignore') as f:
        text = f.read()
    return text

def load_text_from_csv(csv_file_path, delimiter=','):
        try:
            with open(csv_file_path, 'rb') as f:
                raw_data = f.read()
                result = chardet.detect(raw_data)
                encoding = result['encoding']
            with open(csv_file_path, 'r', encoding=encoding, errors='ignore') as f:
                lines = [line.strip() for line in f]

            temp_file = "temp.csv"
            with open(temp_file, 'w', encoding=encoding, errors='ignore') as f:
                f.write('\n'.join(lines))

            try:
                # Попробуем прочитать с предположением, что нет заголовков
                df = pd.read_csv(temp_file, encoding=encoding, on_bad_lines='skip',
                                 delimiter=delimiter, header=None,
                                 skipinitialspace=True)
            except pd.errors.ParserError as e:
                print(f"Ошибка парсинга CSV (header=None): {e}")
                # Попробуем прочитать с предположением, что есть заголовки
                df = pd.read_csv(temp_file, encoding=encoding, on_bad_lines='skip',
                                 delimiter=delimiter, skipinitialspace=True)
            os.remove(temp_file)
        except Exception as e:
            print(f"Ошибка загрузки CSV с кодировкой '{encoding}' или 'latin1': {e}")
            df = pd.read_csv(csv_file_path, encoding='latin1', on_bad_lines='skip', delimiter=delimiter)
        # Объединяем все текстовые столбцы в одну строку
        text_columns = df.select_dtypes(include='object').columns.tolist()
        text = " ".join(df[text_columns].astype(str).fillna('').values.flatten())
        return text

# Загрузка текста
DELIMITER = ','
if os.path.isdir(dataset_path) or dataset_path.endswith(".zip"):
    if os.path.isdir(dataset_path):
        for file in os.listdir(dataset_path):
            if file.endswith(".csv"):
                csv_file_path = os.path.join(dataset_path, file)
                text = load_text_from_csv(csv_file_path, DELIMITER)
                break
    elif dataset_path.endswith(".zip"):
        text = load_text_from_zip(dataset_path)


# ----------------------------------------------------------------------
# Предварительная обработка текста
# ----------------------------------------------------------------------

# Обрабатываем текст
print(f'Длина текста: {len(text)} символов')
print(f'Первые 1000 символов: {text[:1000]}')
vocab = sorted(set(text))
print(f'Количество уникальных символов: {len(vocab)}')

# example_texts = ['abcdefg', 'xyz']
example_texts = text[11:]
print(example_texts[:100])
print(example_texts[-1000:])
chars = tf.strings.unicode_split(example_texts, input_encoding='UTF-8')
print(chars)

ids_from_chars = tf.keras.layers.StringLookup(vocabulary=list(vocab), mask_token=None)
ids = ids_from_chars(chars)

chars_from_ids = tf.keras.layers.StringLookup(vocabulary=ids_from_chars.get_vocabulary(), invert=True, mask_token=None)
chars = chars_from_ids(ids)



all_ids = ids_from_chars(tf.strings.unicode_split(text, 'UTF-8'))
print(all_ids)

ids_dataset = tf.data.Dataset.from_tensor_slices(all_ids)
seq_length = int(100/2)
sequences = ids_dataset.batch(seq_length + 1, drop_remainder=True)

def split_input_target(sequence):
    input_text = sequence[:-1]
    target_text = sequence[1:]
    return input_text, target_text

def text_from_ids(ids):
  return tf.strings.reduce_join(chars_from_ids(ids), axis=-1)

dataset = sequences.map(split_input_target)


BATCH_SIZE = int(64/2)
BUFFER_SIZE = 10000
dataset = (dataset.shuffle(BUFFER_SIZE).batch(BATCH_SIZE, drop_remainder=True).prefetch(tf.data.experimental.AUTOTUNE))

vocab_size = len(ids_from_chars.get_vocabulary())
embedding_dim = int(256/2)
rnn_units = int(1024/2)

# ----------------------------------------------------------------------
# Модель
# ----------------------------------------------------------------------

class MyModel(tf.keras.Model):
  def __init__(self, vocab_size, embedding_dim, rnn_units):
    super().__init__()
    self.embedding = tf.keras.layers.Embedding(vocab_size, embedding_dim)
    self.gru = tf.keras.layers.GRU(rnn_units,
                                   return_sequences=True,
                                   return_state=True)
    self.dense = tf.keras.layers.Dense(vocab_size)

  def call(self, inputs, states=None, return_state=False, training=False):
    x = inputs
    x = self.embedding(x, training=training)
    if states is None:
      states = self.gru.get_initial_state(tf.shape(x)[0])
    x, states = self.gru(x, initial_state=states, training=training)
    x = self.dense(x, training=training)

    if return_state:
      return x, states
    else:
      return x

# ----------------------------------------------------------------------
# Обучение и генерация текста
# ----------------------------------------------------------------------

# Функция для обучения модели и генерации текста
def train_and_generate(model, chars_from_ids, ids_from_chars, epochs, loss_fn, optimizer_class, gen_length, temperature):
    # Создаем новый экземпляр оптимизатора для каждой итерации
    optimizer = optimizer_class()
    # optimizer = tf.keras.optimizers.Adam()

    try:
        model.compile(optimizer=optimizer, loss=loss_fn)
    except Exception as e:
        print(f"Ошибка компиляции модели с {loss_fn.__name__} функцией потерь и {optimizer.__class__.__name__} оптимизатором: {e}")
        return

    checkpoint_dir = './training_checkpoints'
    checkpoint_prefix = os.path.join(checkpoint_dir, "ckpt_{epoch}.weights.h5")
    checkpoint_callback = tf.keras.callbacks.ModelCheckpoint(
        filepath=checkpoint_prefix,
        save_weights_only=True)

    model.fit(dataset, epochs=epochs, callbacks=[checkpoint_callback])


    one_step_model = OneStep(model, chars_from_ids, ids_from_chars, temperature)

    start = time.time()
    states = None
    next_char = tf.constant(['How are you? ']) # Начальный текст
    result = [next_char]

    for n in range(gen_length):
        next_char, states = one_step_model.generate_one_step(next_char, states=states)
        result.append(next_char)

    result = tf.strings.join(result)
    end = time.time()

    print("="*80)
    print(f"Результаты обучения и генерации:")
    print(f"  Количество эпох: {epochs}")
    # print(f"  Функция потерь: {loss_fn.__name__}")
    print(f"  Оптимизатор: {optimizer.__class__.__name__}")
    print(f"  Длина сгенерированного текста: {gen_length} символов")
    print(f"  Температура: {temperature}")
    print(f"  Сгенерированный текст:\n{result[0].numpy().decode('utf-8')}")
    print("="*80)
    print(f'Время выполнения: {end - start:.2f} секунд\n')

# ----------------------------------------------------------------------
# Генерация текста пошагово
# ----------------------------------------------------------------------

class OneStep(tf.keras.Model):
  def __init__(self, model, chars_from_ids, ids_from_chars, temperature=1.0):
    super().__init__()
    self.temperature = temperature
    self.model = model
    self.chars_from_ids = chars_from_ids
    self.ids_from_chars = ids_from_chars

    # Маска для предотвращения генерации "[UNK]"
    skip_ids = self.ids_from_chars(['[UNK]'])[:, None]
    sparse_mask = tf.SparseTensor(
        values=[-float('inf')]*len(skip_ids),
        indices=skip_ids,
        dense_shape=[len(ids_from_chars.get_vocabulary())])
    self.prediction_mask = tf.sparse.to_dense(sparse_mask)

  @tf.function
  def generate_one_step(self, inputs, states=None):
    # Преобразуем строки в ID токенов
    input_chars = tf.strings.unicode_split(inputs, 'UTF-8')
    input_ids = self.ids_from_chars(input_chars).to_tensor()

    # Запускаем модель
    predicted_logits, states = self.model(inputs=input_ids, states=states,
                                          return_state=True)
    # Используем только последнее предсказание
    predicted_logits = predicted_logits[:, -1, :]
    predicted_logits = predicted_logits/self.temperature
    # Применяем маску для предотвращения генерации "[UNK]"
    predicted_logits = predicted_logits + self.prediction_mask

    # Сэмплируем выходные логиты для генерации ID токенов
    predicted_ids = tf.random.categorical(predicted_logits, num_samples=1)
    predicted_ids = tf.squeeze(predicted_ids, axis=-1)

    # Преобразуем ID токенов в символы
    predicted_chars = self.chars_from_ids(predicted_ids)

    # Возвращаем символы и состояние модели
    return predicted_chars, states

# ----------------------------------------------------------------------
# Параметры экспериментов
# ----------------------------------------------------------------------

# Список количества эпох
EPOCHS_LIST = [5, 15]

# Список длин генерируемого текста
GEN_LENGTHS = [100, 1000]


# Список оптимизаторов
OPTIMIZERS = [
    tf.keras.optimizers.Adam,
    tf.keras.optimizers.SGD,
    tf.keras.optimizers.RMSprop
]

TEMPERATURES = [1.0]

LOSS_FUNCTION = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True) # Определение функции потерь здесь


# Запускаем циклы обучения и генерации
for epochs in EPOCHS_LIST:
    print(f"Обучение с эпохами: {epochs}")
    # Создаем новую модель перед каждым набором эпох
    model = MyModel(
        vocab_size=vocab_size,
        embedding_dim=embedding_dim,
        rnn_units=rnn_units
    )
    # Создаем новые StringLookup слои
    ids_from_chars = tf.keras.layers.StringLookup(vocabulary=list(vocab), mask_token=None)
    chars_from_ids = tf.keras.layers.StringLookup(vocabulary=ids_from_chars.get_vocabulary(), invert=True, mask_token=None)


    for optimizer_class in OPTIMIZERS:
        for gen_length in GEN_LENGTHS:
            for temperature in TEMPERATURES:
                train_and_generate(model, chars_from_ids, ids_from_chars, epochs, LOSS_FUNCTION, optimizer_class, gen_length, temperature)


print(f"\n\nWarning: Please check the CSV file. pandas skipped some lines because the number of fields was not correct. These lines may contain data you want. It is advised that you fix the CSV data to not skip any lines for best results.\n")


Kagglehub updated successfully.
Путь к файлам датасета: /root/.cache/kagglehub/datasets/trainingdatapro/generated-e-mail-spam/versions/1
Длина текста: 24053 символов
Первые 1000 символов: Title	Text The Love-Booster Crew" Unbelievable Simple Trick to Help Make Your Life Easier	"Yo! What's up! Do you ever feel like you just can't seem to catch a break? Are you always drowning in work and responsibilities? BOB'S MAGIC is ridiculously simple to use and helps you smash through your to-do list in no time. Say goodbye to stress and hello to productivity with this game-changing trick! The BOB'S MAGIC Crew" Find your ride or die today and have an epic summer full of unforgettable memories! The YourLuv4Ever.com Crew" Visit <LINK> today to learn more about our program and how it can work for you. Take the first step towards a brighter financial future. Don't miss out on this incredible opportunity! Sign up now for a free consultation and start your journey towards financial freedom. The EraseDeb

**Результаты обучения и генерации:**

*   **Сгенерированный текст:** Текст не всегда выглядит
как осмысленный текст, местами имеет вид набора случайных символов.
*   **Время выполнения:** меняется в зависимости от параметров.
*   **Оптимизаторы:**
  *   **Adam** показал неоднозначные результаты, в некоторых случаях генерируя мусорный текст, а в некоторых случаях — более структурированный.
  *   **SGD** в основном генерирует пустые строки.
  *   **RMSprop** показывает не очень хорошие результаты, как и **SGD**.
*   **Количество эпох:** На большем количестве эпох модель не всегда показывает лучшие результаты.

-------------------

**Выводы по обучению:**

*   **Необходима более глубокая настройка параметров:** Параметры обучения и модели (количество эпох, размерности, оптимизаторы),вероятно, не являются оптимальными. Необходимо более тщательно подбирать гиперпараметры для обучения модели.
*   **Необходим более качественный набор данных:** Набор данных имеет много пропусков и ошибок в структуре данных, что может сказаться на конечном результате.
*   **Генерация текста несовершенна:** Модель с текущими параметрами и данными не генерирует осмысленный текст, показывая в основном набор случайных символов.
-------------------

**Необходима дополнительная работа над:**

*   Улучшением качества данны
*   Тщательной настройкой гиперпараметров модели и параметров обучения
*   Более глубоким анализом результатов обучения
*   Использование более сложных архитектур нейронных сетей

---------------------------------------------------------------------------------

In [ ]:
#для обучения с CategoricalCrossentropy

import tensorflow as tf
import numpy as np
import os, sys
import time
import kagglehub
import zipfile
import io
import pandas as pd
import chardet
import csv
import subprocess

# Обновление kagglehub (необязательно)
try:
    subprocess.check_call(['pip', 'install', '--upgrade', 'kagglehub'])
    print("Kagglehub updated successfully.")
except subprocess.CalledProcessError as e:
    print(f"Failed to update kagglehub: {e}")

# ----------------------------------------------------------------------
# Загрузка данных
# ----------------------------------------------------------------------
# Загрузка датасета с помощью kagglehub
dataset_path = kagglehub.dataset_download("trainingdatapro/generated-e-mail-spam")
print("Путь к файлам датасета:", dataset_path)

# Функция для загрузки текста из zip-архива
def load_text_from_zip(zip_file_path):
    with zipfile.ZipFile(zip_file_path, 'r') as zip_file:
        # Получаем имя первого CSV-файла в zip-архиве
        text_file_name = None
        for name in zip_file.namelist():
            if name.endswith(".csv"):
                text_file_name = name
                break
        if not text_file_name:
            raise Exception("CSV-файл не найден в zip-архиве")

        with zip_file.open(text_file_name, 'r') as f:
            df = pd.read_csv(f, encoding = 'latin1')
            # Объединяем все текстовые столбцы в одну строку
            text_columns = df.select_dtypes(include='object').columns.tolist()
            text = " ".join(df[text_columns].astype(str).fillna('').values.flatten())
    return text

# Функция для загрузки текста из локального текстового файла
def load_text_from_file(file_path):
    with open(file_path, 'r', encoding='utf-8', errors='ignore') as f:
        text = f.read()
    return text

def load_text_from_csv(csv_file_path, delimiter=','):
        try:
            with open(csv_file_path, 'rb') as f:
                raw_data = f.read()
                result = chardet.detect(raw_data)
                encoding = result['encoding']
            with open(csv_file_path, 'r', encoding=encoding, errors='ignore') as f:
                lines = [line.strip() for line in f]

            temp_file = "temp.csv"
            with open(temp_file, 'w', encoding=encoding, errors='ignore') as f:
                f.write('\n'.join(lines))

            try:
                # Попробуем прочитать с предположением, что нет заголовков
                df = pd.read_csv(temp_file, encoding=encoding, on_bad_lines='skip',
                                 delimiter=delimiter, header=None,
                                 skipinitialspace=True)
            except pd.errors.ParserError as e:
                print(f"Ошибка парсинга CSV (header=None): {e}")
                # Попробуем прочитать с предположением, что есть заголовки
                df = pd.read_csv(temp_file, encoding=encoding, on_bad_lines='skip',
                                 delimiter=delimiter, skipinitialspace=True)
            os.remove(temp_file)
        except Exception as e:
            print(f"Ошибка загрузки CSV с кодировкой '{encoding}' или 'latin1': {e}")
            df = pd.read_csv(csv_file_path, encoding='latin1', on_bad_lines='skip', delimiter=delimiter)
        # Объединяем все текстовые столбцы в одну строку
        text_columns = df.select_dtypes(include='object').columns.tolist()
        text = " ".join(df[text_columns].astype(str).fillna('').values.flatten())
        return text

# Загрузка текста
DELIMITER = ','
if os.path.isdir(dataset_path) or dataset_path.endswith(".zip"):
    if os.path.isdir(dataset_path):
        for file in os.listdir(dataset_path):
            if file.endswith(".csv"):
                csv_file_path = os.path.join(dataset_path, file)
                text = load_text_from_csv(csv_file_path, DELIMITER)
                break
    elif dataset_path.endswith(".zip"):
        text = load_text_from_zip(dataset_path)


# ----------------------------------------------------------------------
# Предварительная обработка текста
# ----------------------------------------------------------------------

# Обрабатываем текст
print(f'Длина текста: {len(text)} символов')
print(f'Первые 1000 символов: {text[:1000]}')
vocab = sorted(set(text))
print(f'Количество уникальных символов: {len(vocab)}')

# example_texts = ['abcdefg', 'xyz']
example_texts = text[11:]
print(example_texts[:100])
print(example_texts[-1000:])
chars = tf.strings.unicode_split(example_texts, input_encoding='UTF-8')
print(chars)

ids_from_chars = tf.keras.layers.StringLookup(vocabulary=list(vocab), mask_token=None)
ids = ids_from_chars(chars)

chars_from_ids = tf.keras.layers.StringLookup(vocabulary=ids_from_chars.get_vocabulary(), invert=True, mask_token=None)
chars = chars_from_ids(ids)



all_ids = ids_from_chars(tf.strings.unicode_split(text, 'UTF-8'))
print(all_ids)

ids_dataset = tf.data.Dataset.from_tensor_slices(all_ids)
seq_length = int(100/2)
sequences = ids_dataset.batch(seq_length + 1, drop_remainder=True)

def split_input_target(sequence):
    input_text = sequence[:-1]
    target_text = sequence[1:]
    return input_text, target_text

def text_from_ids(ids):
  return tf.strings.reduce_join(chars_from_ids(ids), axis=-1)

dataset = sequences.map(split_input_target)


BATCH_SIZE = int(64/2)
BUFFER_SIZE = 10000
dataset = (dataset.shuffle(BUFFER_SIZE).batch(BATCH_SIZE, drop_remainder=True).prefetch(tf.data.experimental.AUTOTUNE))

vocab_size = len(ids_from_chars.get_vocabulary())
embedding_dim = int(256/2)
rnn_units = int(1024/2)

# ----------------------------------------------------------------------
# Модель
# ----------------------------------------------------------------------

class MyModel(tf.keras.Model):
  def __init__(self, vocab_size, embedding_dim, rnn_units):
    super().__init__()
    self.embedding = tf.keras.layers.Embedding(vocab_size, embedding_dim)
    self.gru = tf.keras.layers.GRU(rnn_units,
                                   return_sequences=True,
                                   return_state=True)
    self.dense = tf.keras.layers.Dense(vocab_size)

  def call(self, inputs, states=None, return_state=False, training=False):
    x = inputs
    x = self.embedding(x, training=training)
    if states is None:
      states = self.gru.get_initial_state(tf.shape(x)[0])
    x, states = self.gru(x, initial_state=states, training=training)
    x = self.dense(x, training=training)

    if return_state:
      return x, states
    else:
      return x

# ----------------------------------------------------------------------
# Обучение и генерация текста
# ----------------------------------------------------------------------

# Функция для обучения модели и генерации текста
def train_and_generate(model, chars_from_ids, ids_from_chars, epochs, loss_fn, optimizer_class, gen_length, temperature):
    # Создаем новый экземпляр оптимизатора для каждой итерации
    optimizer = optimizer_class()
    # optimizer = tf.keras.optimizers.Adam()

    try:
        model.compile(optimizer=optimizer, loss=loss_fn, metrics=['accuracy'])
    except Exception as e:
        print(f"Ошибка компиляции модели с {loss_fn.__name__} функцией потерь и {optimizer.__class__.__name__} оптимизатором: {e}")
        return

    checkpoint_dir = './training_checkpoints'
    checkpoint_prefix = os.path.join(checkpoint_dir, "ckpt_{epoch}.weights.h5")
    checkpoint_callback = tf.keras.callbacks.ModelCheckpoint(
        filepath=checkpoint_prefix,
        save_weights_only=True)

    model.fit(dataset.map(lambda x, y: (x, tf.one_hot(y, depth=vocab_size))),
                  epochs=epochs, callbacks=[checkpoint_callback])


    one_step_model = OneStep(model, chars_from_ids, ids_from_chars, temperature)

    start = time.time()
    states = None
    next_char = tf.constant(['How are you? ']) # Начальный текст
    result = [next_char]

    for n in range(gen_length):
        next_char, states = one_step_model.generate_one_step(next_char, states=states)
        result.append(next_char)

    result = tf.strings.join(result)
    end = time.time()

    print("="*80)
    print(f"Результаты обучения и генерации:")
    print(f"  Количество эпох: {epochs}")
    #print(f"  Функция потерь: {loss_fn.__name__}")
    print(f"  Оптимизатор: {optimizer.__class__.__name__}")
    print(f"  Длина сгенерированного текста: {gen_length} символов")
    print(f"  Температура: {temperature}")
    print(f"  Сгенерированный текст:\n{result[0].numpy().decode('utf-8')}")
    print("="*80)
    print(f'Время выполнения: {end - start:.2f} секунд\n')

# ----------------------------------------------------------------------
# Генерация текста пошагово
# ----------------------------------------------------------------------

class OneStep(tf.keras.Model):
  def __init__(self, model, chars_from_ids, ids_from_chars, temperature=1.0):
    super().__init__()
    self.temperature = temperature
    self.model = model
    self.chars_from_ids = chars_from_ids
    self.ids_from_chars = ids_from_chars

    # Маска для предотвращения генерации "[UNK]"
    skip_ids = self.ids_from_chars(['[UNK]'])[:, None]
    sparse_mask = tf.SparseTensor(
        values=[-float('inf')]*len(skip_ids),
        indices=skip_ids,
        dense_shape=[len(ids_from_chars.get_vocabulary())])
    self.prediction_mask = tf.sparse.to_dense(sparse_mask)

  @tf.function
  def generate_one_step(self, inputs, states=None):
    # Преобразуем строки в ID токенов
    input_chars = tf.strings.unicode_split(inputs, 'UTF-8')
    input_ids = self.ids_from_chars(input_chars).to_tensor()

    # Запускаем модель
    predicted_logits, states = self.model(inputs=input_ids, states=states,
                                          return_state=True)
    # Используем только последнее предсказание
    predicted_logits = predicted_logits[:, -1, :]
    predicted_logits = predicted_logits/self.temperature
    # Применяем маску для предотвращения генерации "[UNK]"
    predicted_logits = predicted_logits + self.prediction_mask

    # Сэмплируем выходные логиты для генерации ID токенов
    predicted_ids = tf.random.categorical(predicted_logits, num_samples=1)
    predicted_ids = tf.squeeze(predicted_ids, axis=-1)


    # Преобразуем ID токенов в символы
    predicted_chars = self.chars_from_ids(predicted_ids)

    # Возвращаем символы и состояние модели
    return predicted_chars, states

# ----------------------------------------------------------------------
# Параметры экспериментов
# ----------------------------------------------------------------------

# Список количества эпох
EPOCHS_LIST = [5, 15]

# Список длин генерируемого текста
GEN_LENGTHS = [100, 1000]


# Список оптимизаторов
OPTIMIZERS = [
    tf.keras.optimizers.Adam,
    tf.keras.optimizers.SGD,
    tf.keras.optimizers.RMSprop
]

TEMPERATURES = [1.0]

LOSS_FUNCTION = tf.keras.losses.CategoricalCrossentropy # Определение функции потерь здесь


# Запускаем циклы обучения и генерации
for epochs in EPOCHS_LIST:
    print(f"Обучение с эпохами: {epochs}")
    # Создаем новую модель перед каждым набором эпох
    model = MyModel(
        vocab_size=vocab_size,
        embedding_dim=embedding_dim,
        rnn_units=rnn_units
    )
    # Создаем новые StringLookup слои
    ids_from_chars = tf.keras.layers.StringLookup(vocabulary=list(vocab), mask_token=None)
    chars_from_ids = tf.keras.layers.StringLookup(vocabulary=ids_from_chars.get_vocabulary(), invert=True, mask_token=None)


    for optimizer_class in OPTIMIZERS:
        for gen_length in GEN_LENGTHS:
            for temperature in TEMPERATURES:
                train_and_generate(model, chars_from_ids, ids_from_chars, epochs, LOSS_FUNCTION, optimizer_class, gen_length, temperature)


print(f"\n\nWarning: Please check the CSV file. pandas skipped some lines because the number of fields was not correct. These lines may contain data you want. It is advised that you fix the CSV data to not skip any lines for best results.\n")


Kagglehub updated successfully.
Путь к файлам датасета: /root/.cache/kagglehub/datasets/trainingdatapro/generated-e-mail-spam/versions/1
Длина текста: 24053 символов
Первые 1000 символов: Title	Text The Love-Booster Crew" Unbelievable Simple Trick to Help Make Your Life Easier	"Yo! What's up! Do you ever feel like you just can't seem to catch a break? Are you always drowning in work and responsibilities? BOB'S MAGIC is ridiculously simple to use and helps you smash through your to-do list in no time. Say goodbye to stress and hello to productivity with this game-changing trick! The BOB'S MAGIC Crew" Find your ride or die today and have an epic summer full of unforgettable memories! The YourLuv4Ever.com Crew" Visit <LINK> today to learn more about our program and how it can work for you. Take the first step towards a brighter financial future. Don't miss out on this incredible opportunity! Sign up now for a free consultation and start your journey towards financial freedom. The EraseDeb

**Результаты обучения и генерации:**

*   **Функция потерь (Loss):** Высокие значения потерь (loss) в течение всего обучения свидетельствуют о том, что модель не может предсказывать выход с высокой точностью.
*   **Точность (Accuracy):** Очень низкая точность, близка к нулю или очень мала - модель не может правильно определить следующий символ в последовательности.
*   **Генерация текста:** Сгенерированный текст состоит из повторяющихся символов, пробелов или пустых строк: модель не "выучила" структуру текста. Она либо выдаёт случайный набор символов, либо зацикливается на одном, выдавая бесконечное повторение.
*   **Оптимизаторы:** Результаты обучения сильно зависят от выбранного оптимизатора, ни один из оптимизаторов не показал приемлемых результатов.
  * **Adam:** Показывает некоторую "диверсификацию" генерируемого текста, но всё равно не генерирует осмысленные слова.
  * **SGD:** Практически всегда выдаёт повторяющиеся символы или пустой текст.
  * **RMSprop:** Как и **SGD**, не может научиться генерировать осмысленный текст.
-------------------
**Выводы по обучению:**
*   **Модель не обучается:** Основная проблема заключается в том, что модель не обучается должным образом. Показатели loss и accuracy остаются плохими на протяжении всего процесса обучения
*   **Проблема с оптимизацией:** Вероятно выбранные оптимизаторы с параметрами по умолчанию не приводят к сходимости модели
*   **Архитектура модели:** Нейронная сеть может быть слишком простой для такой задачи
*   **Предварительная обработка:** Методы обработки данных ,вероятно, недостаточно хороши, стоит пробовать другие подходы к предобработке текста, в том числе токенизацию на основе слов

-------------------
**Необходима дополнительная работа над:**
*   **Моделью:**
  *   **Эксперименты с гиперпараметрами:** Подобрать оптимальные значения для количества эпох, размера эмбеддинга, количества RNN-юнитов, размера батча и т.д.
  *   **Попробовать другие архитектуры:** Использовать LSTM вместо GRU, добавить больше слоев, попробовать механизм "внимания"
  *   **Нормализация:** Batch Normalization, Dropout
*   **Обучением:**
  *   **Оптимизация функции потерь:** Проверить корректность one-hot кодирования
  *   **Подбор оптимизаторов:** Попробовать другие оптимизаторы и использовать Learning Rate Scheduler
  *   **Мониторинг обучения:** Добавить валидационный набор, отслеживать loss и accuracy на обучающем и валидационном наборах